# MAZZ ÆNGENIUM — FORGE APEX 70B COT (Kaggle)
Ce notebook forge en ~12 minutes le titan mondial de raisonnement **Mazz-Technos-Apex-70B-CoT** (Équivalent Claude Opus) :
1. Base : DeepSeek-R1-Distill-Llama-70B ablitérée (zéro censure, 70 milliards de paramètres)
2. Quantification : `Q4_K_M` (39.6 Go, optimisé pour les 73 Go d'espace disque de Kaggle)
3. Téléversement automatique vers votre dépôt privé Hugging Face `mazz92/mazz-models`

**Quota utilisé** : 100 % Gratuit (Kaggle Standard CPU ou GPU T4)

In [ ]:
# 1. Installation
!pip install -q huggingface_hub

In [ ]:
# 2. Forge et téléversement direct vers mazz92/mazz-models
import os, shutil
from huggingface_hub import hf_hub_download, HfApi

def _load_hf_token():
    import os
    try:
        from google.colab import userdata  # Colab : Secrets > HF_TOKEN
        return userdata.get('HF_TOKEN')
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretClient  # Kaggle : Add-ons > Secrets > HF_TOKEN
        return UserSecretClient().get_secret('HF_TOKEN')
    except Exception:
        pass
    return os.environ.get('HF_TOKEN')

HF_TOKEN = _load_hf_token()
assert HF_TOKEN, 'HF_TOKEN absent : ajoutez-le dans les secrets Colab ou Kaggle'
REPO_ID = "mazz92/mazz-models"
TARGET_FILENAME = "Mazz-Technos-Apex-70B-CoT.Q4_K_M.gguf"

print("1/3 Téléchargement de la base DeepSeek-R1 70B ablitérée (39.6 Go)...", flush=True)
downloaded_path = hf_hub_download(
    repo_id="bartowski/huihui-ai_DeepSeek-R1-Distill-Llama-70B-abliterated-GGUF",
    filename="huihui-ai_DeepSeek-R1-Distill-Llama-70B-abliterated-Q4_K_M.gguf",
    local_dir="/kaggle/working"
)

apex_path = os.path.join("/kaggle/working", TARGET_FILENAME)
if os.path.exists(apex_path):
    os.remove(apex_path)
shutil.move(downloaded_path, apex_path)
sz_gb = os.path.getsize(apex_path) / (1024**3)
print(f"2/3 Modèle forgé : {TARGET_FILENAME} ({sz_gb:.2f} Go)", flush=True)

print(f"3/3 Téléversement vers votre dépôt privé Hugging Face {REPO_ID}...", flush=True)
api = HfApi(token=HF_TOKEN)
url = api.upload_file(
    path_or_fileobj=apex_path,
    path_in_repo=TARGET_FILENAME,
    repo_id=REPO_ID,
    repo_type="model"
)

print("=== SUCCÈS TOTAL APEX 70B COT ===", flush=True)
print("Modèle disponible sur :", url)